In [38]:
!pip install emoji pyarabic nltk

In [39]:
## Preprocessing and Cleaning:
import nltk
import re
import emoji
import pandas as pd
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize

## Text Representation:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


nltk.download('punkt')        # ---------> 'punkt' is a pre-trained NLTK tokenizer model that enables correct splitting of text into words or sentences.
nltk.download('stopwords')    # ---------> 'stopwords' is an NLTK resource that provides lists of common words (like “the”, “and”) to filter out during text preprocessing.
nltk.download('punkt_tab')    # ---------> 'punkt_tab' is a pre-trained NLTK tokenizer model that allows splitting text into words or sentences for tokenization.

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [40]:
df=pd.read_csv('AAFAQ_Dataset.csv')
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


### **Exploratory Data Analysis**

In [41]:
#Dataset size
df.size

15027

In [42]:
#Dataset shape
df.shape

(5009, 3)

In [43]:
#olumn names
df.columns

Index(['QuestionText', 'Category', 'Answer'], dtype='object')

In [44]:
#Data types
df.dtypes

,0
QuestionText,object
Category,object
Answer,object


In [45]:
#Labels
df['Category'].unique()

array(['التعليم', 'الاقتصاد والعمل', 'الصحة', 'التكنولوجيا',
       'البيئة والطاقة', 'الدين', 'الثقافة', 'الجغرافيا',
       'السياسة والقانون', 'التاريخ', 'العلوم', 'الترفيه', 'الرياضة',
       'السفر والسياحة', 'التطوع', 'علم الاجتماع', 'البيولوجيا'],
      dtype=object)

In [46]:
#Null values check
df.isnull().sum()

,0
QuestionText,0
Category,0
Answer,0


In [47]:
#Duplicates Check
df.duplicated().sum()

np.int64(188)

In [48]:
#Duplicated questions
df.duplicated(subset=["QuestionText"]).sum()

np.int64(364)

In [49]:
#Duplicated answers
df.duplicated(subset=["Answer"]).sum()

np.int64(415)

In [50]:
#Duplicates dropping
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [51]:
df.duplicated(subset=["QuestionText"]).sum()

np.int64(176)

In [52]:
df.duplicated(subset=["Answer"]).sum()

np.int64(227)

In [53]:
duplicates = df.groupby('QuestionText')['Answer'].nunique()

conflicting = duplicates[duplicates > 1].index

result = df[df['QuestionText'].isin(conflicting)].sort_values('QuestionText')

print(f"Number of questions with conflicting answers: {len(conflicting)}")
print()
print(result[['QuestionText', 'Answer']])

Number of questions with conflicting answers: 135

                                          QuestionText  \
1079       أعرب عن وجهة نظرك في دور المرأة في المجتمع.   
1099       أعرب عن وجهة نظرك في دور المرأة في المجتمع.   
3354  أي الصحابة قادوا الجيوش الإسلامية في حروب الردة؟   
3156  أي الصحابة قادوا الجيوش الإسلامية في حروب الردة؟   
3657                        أين تقع الأهرامات المصرية؟   
...                                                ...   
128                      هل استطاع المسلمون فتح القدس؟   
126           هل تمت توسعة المسجد النبوي في عهد عثمان؟   
124           هل تمت توسعة المسجد النبوي في عهد عثمان؟   
125          هل جمع زيد بن ثابت القرآن في عهد أبي بكر؟   
123          هل جمع زيد بن ثابت القرآن في عهد أبي بكر؟   

                                                 Answer  
1079  أرى أن دور المرأة في المجتمع حيوي وأساسي، فهي ...  
1099  أرى أن المرأة تلعب دوراً حيوياً في تنمية المجت...  
3354  أرسل الجيوش الإسلاميّة بقيادة عكرمة بن أبي جهل...  
3156  أرسل الجيوش ال

In [54]:
#label distribution
label_counts = df['Category'].value_counts()
print(label_counts)
print(f"\nTotal categories: {label_counts.shape[0]}")

Category
الثقافة             725
التعليم             600
العلوم              597
الصحة               537
البيئة والطاقة      500
التكنولوجيا         439
التاريخ             402
الاقتصاد والعمل     290
الجغرافيا           203
السياسة والقانون    194
الدين               114
الرياضة              67
البيولوجيا           50
الترفيه              37
السفر والسياحة       33
التطوع               25
علم الاجتماع          8
Name: count, dtype: int64

Total categories: 17


In [55]:
from collections import defaultdict

grouped = df.groupby('QuestionText')

conflict_count = 0

for question, group in grouped:
    unique_labels = group['Category'].unique()

    if len(unique_labels) > 1:
        conflict_count += 1
        print("=" * 80)
        print(f"Question: {question}")
        print(f"Labels: {list(unique_labels)}")

        label_counts = group['Category'].value_counts()
        print(f"Label counts:")
        for label, count in label_counts.items():
            print(f"   {label}: {count} ({'MAJORITY' if count == label_counts.max() else ''})")
        print(f"   Majority label: {label_counts.idxmax()}")

        unique_answers = group['Answer'].unique()

        if len(unique_answers) == 1:
            print(f"Answers are the SAME:")
            print(f"   {unique_answers[0]}")
        else:
            print(f"Answers are DIFFERENT:")
            for _, row in group.iterrows():
                print(f"   [{row['Category']}] {row['Answer']}")

        print()

print("=" * 80)
print(f"Total conflicting questions: {conflict_count}")

Question: أين تقع الأهرامات المصرية؟
Labels: ['التاريخ', 'الجغرافيا']
Label counts:
   التاريخ: 1 (MAJORITY)
   الجغرافيا: 1 (MAJORITY)
   Majority label: التاريخ
Answers are DIFFERENT:
   [التاريخ] الأهرامات المصرية تقع في منطقة الجيزة بالقرب من القاهرة.
   [الجغرافيا] الأهرامات المصرية تقع في الجيزة بالقرب من القاهرة.

Question: إلامَ تشير النتائج الأولية للبحث؟
Labels: ['التعليم', 'العلوم']
Label counts:
   التعليم: 1 (MAJORITY)
   العلوم: 1 (MAJORITY)
   Majority label: التعليم
Answers are DIFFERENT:
   [التعليم] تشير النتائج الأولية للبحث إلى وجود علاقة إيجابية بين استخدام التكنولوجيا الحديثة وزيادة تحصيل الطلاب الأكاديمي.
   [العلوم] تشير النتائج الأولية للبحث إلى وجود علاقة بين المتغيرات المدروسة، مما يتطلب مزيداً من التحليل لتأكيد النتائج.

Question: احسبْ الزمن اللازم لقطع مسافة 100 كم بسرعة 50 كم/ساعة.
Labels: ['التعليم', 'العلوم']
Label counts:
   التعليم: 1 (MAJORITY)
   العلوم: 1 (MAJORITY)
   Majority label: التعليم
Answers are DIFFERENT:
   [التعليم] الزمن اللازم لقطع مس

In [56]:
conflicting_questions = df.groupby('QuestionText')['Category'].nunique()
conflicting_idx = conflicting_questions[conflicting_questions > 1].index

df = df[~df['QuestionText'].isin(conflicting_idx)]


In [57]:
Columns = ['QuestionText', 'Answer']

In [58]:
label=df['Category']

In [59]:
df['QuestionText'].apply(type).value_counts()

,count
QuestionText,
<class 'str'>,4776


In [60]:
df['Answer'].apply(type).value_counts()

,count
Answer,
<class 'str'>,4776


In [61]:
has_english_q = df['QuestionText'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_q

np.int64(22)

In [62]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(132)

In [63]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(128)

In [64]:
has_numbers_a = df['Answer'].str.contains(r'\d', regex=True).sum()
has_numbers_a

np.int64(554)

# **Emoji Removal**

In [65]:
import emoji

def has_emoji(text):
    return any(char in emoji.EMOJI_DATA for char in str(text))

emoji_q = df[df['QuestionText'].apply(has_emoji)]
emoji_a = df[df['Answer'].apply(has_emoji)]

print(f"Questions with emojis: {len(emoji_q)}")
print(f"Answers with emojis: {len(emoji_a)}")

Questions with emojis: 0
Answers with emojis: 0


# **Translate English text**

In [66]:
!pip install pyarabic deep_translator

In [67]:
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source='en', target='ar')

def translate_english_to_arabic(text):
    english_words = re.findall(r'[A-Za-z]+', text)
    for word in english_words:
        try:
          translated=translator.translate(word)
          text= text.replace(word,translated)
        except:
          continue
    return text
'''
for col in Columns:
    df[col] = df[col].apply(translate_english_to_arabic)
df
'''
df['QuestionText'] = df['QuestionText'].apply(translate_english_to_arabic)
df['Answer'] = df['Answer'].apply(translate_english_to_arabic)
df

/tmp/ipykernel_390/873698933.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QuestionText'] = df['QuestionText'].apply(translate_english_to_arabic)
/tmp/ipykernel_390/873698933.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Answer'] = df['Answer'].apply(translate_english_to_arabic)


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.
...,...,...,...
5004,ماذا يحدث عند تسخين الحديد؟,العلوم,عند تسخين الحديد، تتسارع حركة جزيئاته مما يؤدي...
5005,ما هو الهدف من اتفاقية التجارة الحرة لأمريكا ا...,الاقتصاد والعمل,الهدف من اتفاقية التجارة الحرة لأمريكا الشمالي...
5006,ما هو الغرض من اتفاقية التجارة الحرة لأمريكا ا...,الاقتصاد والعمل,الغرض من اتفاقية التجارة الحرة لأمريكا الشمالي...
5007,ما هو هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,الدين,يؤدي الإعجاز غرضين رئيسين الأول وهو أثبات أصال...


In [68]:
remaining_q = df[df['QuestionText'].str.contains(r'[A-Za-z]', regex=True)]
remaining_a = df[df['Answer'].str.contains(r'[A-Za-z]', regex=True)]

remaining_q
remaining_a

,QuestionText,Category,Answer
2200,كيف يعمل محرك أقراص الحالة الصلبة؟,التكنولوجيا,يعمل محرك أقراص الحالة الصلبة (SSD) عن طريق تخ...
2207,كيف يتم تشخيص الفصام؟,الصحة,يتم تشخيص الفصام من خلال تقييم الأعراض النفسية...
2313,اشرح كيف تعمل الحاسوب.,التكنولوجيا,الحاسوب يعمل عبر مجموعة من المكونات الأساسية م...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...
2967,ما هو عدد سكان مركز تكس؟,الجغرافيا,عدد سكان مركز تكس (تكس) يعتمد على الموقع الجغر...
3439,عدد بعضًا من شركات إدارة الصيدليات الكبيرة.,الصحة,بعض شركات إدارة الصيدليات الكبيرة تشمل: والجري...
3699,على أي مقياس سيُظهر العلماء قياسات الغطاء النب...,البيئة والطاقة,سيُظهر العلماء قياسات الغطاء النباتي على مقياس...
3812,عدِّدْ مكونات الكمبيوتر.,التكنولوجيا,مكونات الكمبيوتر تشمل: وحدة المعالجة المركزية ...
4080,ما هي وحدة الطاقة الكهربائية؟,العلوم,"وحدة الطاقة الكهربائية هي ""الواط ساعة"" (هل) أو..."
4127,ما هو الافتراض غير المثبت الذي يُعزى عموماً إل...,التكنولوجيا,الافتراض غير المثبت الذي يُعزى عموماً إلى قيمة...


In [69]:
remaining_q

,QuestionText,Category,Answer
551,أي من الأجهزة التالية هو الأفضل للألعاب الإلكت...,التكنولوجيا,من بين الأجهزة الأفضل للألعاب الإلكترونية هو ج...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...


In [70]:
remaining_a

,QuestionText,Category,Answer
2200,كيف يعمل محرك أقراص الحالة الصلبة؟,التكنولوجيا,يعمل محرك أقراص الحالة الصلبة (SSD) عن طريق تخ...
2207,كيف يتم تشخيص الفصام؟,الصحة,يتم تشخيص الفصام من خلال تقييم الأعراض النفسية...
2313,اشرح كيف تعمل الحاسوب.,التكنولوجيا,الحاسوب يعمل عبر مجموعة من المكونات الأساسية م...
2929,كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2...,الاقتصاد والعمل,عدد الشركات المدرجة في بورصة WSE في أغسطس 2009...
2967,ما هو عدد سكان مركز تكس؟,الجغرافيا,عدد سكان مركز تكس (تكس) يعتمد على الموقع الجغر...
3439,عدد بعضًا من شركات إدارة الصيدليات الكبيرة.,الصحة,بعض شركات إدارة الصيدليات الكبيرة تشمل: والجري...
3699,على أي مقياس سيُظهر العلماء قياسات الغطاء النب...,البيئة والطاقة,سيُظهر العلماء قياسات الغطاء النباتي على مقياس...
3812,عدِّدْ مكونات الكمبيوتر.,التكنولوجيا,مكونات الكمبيوتر تشمل: وحدة المعالجة المركزية ...
4080,ما هي وحدة الطاقة الكهربائية؟,العلوم,"وحدة الطاقة الكهربائية هي ""الواط ساعة"" (هل) أو..."
4127,ما هو الافتراض غير المثبت الذي يُعزى عموماً إل...,التكنولوجيا,الافتراض غير المثبت الذي يُعزى عموماً إلى قيمة...


In [71]:
df['QuestionText'][551]

'أي من الأجهزة التالية هو الأفضل للألعاب الإلكترونية: بلاي ستيشن 5، اكس بوكس مسلسل X، ألعاب الكمبيوتر الشخصي؟'

In [72]:
df['QuestionText'][2929]

'كم عدد الشركات المدرجة في بورصة WSE في أغسطس 2009؟'

In [73]:
df['Answer'][2313]

'الحاسوب يعمل عبر مجموعة من المكونات الأساسية مثل وحدة المعالجة المركزية (وحدة المعالجة المركزية)، الذاكرة (كبش)، ووحدات التخزين (الأقراص الصلبة, SSD)، وكل هذه المكونات تعمل معاً من خلال نظام التشغيل لتنفيذ الأوامر ومعالجة البيانات.'

# **Stopwords Removal**

In [74]:
stop_words = set(stopwords.words('arabic'))
stop_words

{'ء',
 'ءَ',
 'آ',
 'آب',
 'آذار',
 'آض',
 'آمينَ',
 'آناء',
 'آنفا',
 'آه',
 'آها',
 'آهاً',
 'آهٍ',
 'آهِ',
 'آي',
 'أ',
 'أبدا',
 'أبريل',
 'أبو',
 'أبٌ',
 'أجل',
 'أجمع',
 'أحد',
 'أخبر',
 'أخذ',
 'أخو',
 'أخٌ',
 'أربع',
 'أربعاء',
 'أربعة',
 'أربعمئة',
 'أربعمائة',
 'أرى',
 'أسكن',
 'أصبح',
 'أصلا',
 'أضحى',
 'أطعم',
 'أعطى',
 'أعلم',
 'أغسطس',
 'أف',
 'أفريل',
 'أفعل به',
 'أفٍّ',
 'أقبل',
 'أقل',
 'أكتوبر',
 'أكثر',
 'أل',
 'ألا',
 'ألف',
 'ألفى',
 'أم',
 'أما',
 'أمام',
 'أمامك',
 'أمامكَ',
 'أمد',
 'أمس',
 'أمسى',
 'أمّا',
 'أن',
 'أنا',
 'أنبأ',
 'أنت',
 'أنتم',
 'أنتما',
 'أنتن',
 'أنتِ',
 'أنشأ',
 'أنى',
 'أنًّ',
 'أنّى',
 'أهلا',
 'أو',
 'أوت',
 'أوشك',
 'أول',
 'أولئك',
 'أولاء',
 'أولالك',
 'أوه',
 'أوّهْ',
 'أى',
 'أي',
 'أيا',
 'أيار',
 'أيضا',
 'أيلول',
 'أين',
 'أينما',
 'أيها',
 'أيّ',
 'أيّان',
 'أُفٍّ',
 'ؤ',
 'إحدى',
 'إذ',
 'إذا',
 'إذاً',
 'إذما',
 'إذن',
 'إزاء',
 'إلا',
 'إلى',
 'إليك',
 'إليكم',
 'إليكما',
 'إليكن',
 'إليكنّ',
 'إليكَ',
 'إلَيْكَ',
 'إلّا',


In [75]:
def remove_sw(text):
  tokens=str(text).split()
  filtered_words = [word for word in tokens if word not in stop_words]
  return " ".join(filtered_words)

for col in Columns:
  df[col]=df[col].apply(remove_sw)
df

/tmp/ipykernel_390/2728529267.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col]=df[col].apply(remove_sw)


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...
1,أليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...
2,أتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً النيتروجين؟,التعليم,الهواء يتكون أساساً النيتروجين.
...,...,...,...
5004,يحدث تسخين الحديد؟,العلوم,تسخين الحديد، تتسارع حركة جزيئاته يؤدي تمدده و...
5005,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية (...
5006,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية (...
5007,هدف إعجاز القرآن وفقًا للمعتقد الإسلامي؟,الدين,يؤدي الإعجاز غرضين رئيسين الأول أثبات أصالة ال...


# **Remove Tashkeel**

In [76]:
#remove tashkeel
for col in Columns:
    df[col] = df[col].apply(lambda x: araby.strip_tashkeel(x))
df

/tmp/ipykernel_390/978524255.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x: araby.strip_tashkeel(x))


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...
1,أليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...
2,أتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا النيتروجين؟,التعليم,الهواء يتكون أساسا النيتروجين.
...,...,...,...
5004,يحدث تسخين الحديد؟,العلوم,تسخين الحديد، تتسارع حركة جزيئاته يؤدي تمدده و...
5005,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية (...
5006,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية (...
5007,هدف إعجاز القرآن وفقا للمعتقد الإسلامي؟,الدين,يؤدي الإعجاز غرضين رئيسين الأول أثبات أصالة ال...


# **Remove Tatweel**

In [77]:
#remove tatweel
for col in Columns:
  df[col]=df[col].apply(lambda x: araby.strip_tatweel(x))
df


/tmp/ipykernel_390/99635122.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col]=df[col].apply(lambda x: araby.strip_tatweel(x))


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...
1,أليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...
2,أتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا النيتروجين؟,التعليم,الهواء يتكون أساسا النيتروجين.
...,...,...,...
5004,يحدث تسخين الحديد؟,العلوم,تسخين الحديد، تتسارع حركة جزيئاته يؤدي تمدده و...
5005,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الهدف اتفاقية التجارة الحرة لأمريكا الشمالية (...
5006,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية؟,الاقتصاد والعمل,الغرض اتفاقية التجارة الحرة لأمريكا الشمالية (...
5007,هدف إعجاز القرآن وفقا للمعتقد الإسلامي؟,الدين,يؤدي الإعجاز غرضين رئيسين الأول أثبات أصالة ال...


# **Normaliza Hamza**

In [78]:
#normalize hamza
for col in Columns:
  df[col]=df[col].apply(lambda x:araby.normalize_hamza(x, method='tasheel'))
df

/tmp/ipykernel_390/2521438801.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col]=df[col].apply(lambda x:araby.normalize_hamza(x, method='tasheel'))


,QuestionText,Category,Answer
0,ايهما افضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعية مصر، ويعد ال...
2,اتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,اتعرف البكتيريا بانها كاينات حية دقيقة؟,التعليم,البكتيريا تعرف بانها كاينات حية دقيقة.
4,ايتكون الهواء اساسا النيتروجين؟,التعليم,الهواء يتكون اساسا النيتروجين.
...,...,...,...
5004,يحدث تسخين الحديد؟,العلوم,تسخين الحديد، تتسارع حركة جزيياته يودي تمدده و...
5005,الهدف اتفاقية التجارة الحرة لامريكا الشمالية؟,الاقتصاد والعمل,الهدف اتفاقية التجارة الحرة لامريكا الشمالية (...
5006,الغرض اتفاقية التجارة الحرة لامريكا الشمالية؟,الاقتصاد والعمل,الغرض اتفاقية التجارة الحرة لامريكا الشمالية (...
5007,هدف اعجاز القران وفقا للمعتقد الاسلامي؟,الدين,يودي الاعجاز غرضين رييسين الاول اثبات اصالة ال...


In [79]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(128)

In [80]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(14)

# **Remove punctuation and english letters left**

In [81]:
#remove punctuation and english letters left
import re
def remove_punct(text):
    cleaned = re.sub(r'[^\u0621-\u064A0-9\s]', '', text)
    return cleaned.strip()

for col in Columns:
  df[col]=df[col].apply(remove_punct)
df

/tmp/ipykernel_390/2449133115.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col]=df[col].apply(remove_punct)


,QuestionText,Category,Answer
0,ايهما افضل الدراسة السابق الوقت الحالي,التعليم,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروة مصر,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعية مصر ويعد الا...
2,اتصعد الشمس الشرق,التعليم,الشمس تصعد الشرق
3,اتعرف البكتيريا بانها كاينات حية دقيقة,التعليم,البكتيريا تعرف بانها كاينات حية دقيقة
4,ايتكون الهواء اساسا النيتروجين,التعليم,الهواء يتكون اساسا النيتروجين
...,...,...,...
5004,يحدث تسخين الحديد,العلوم,تسخين الحديد تتسارع حركة جزيياته يودي تمدده وز...
5005,الهدف اتفاقية التجارة الحرة لامريكا الشمالية,الاقتصاد والعمل,الهدف اتفاقية التجارة الحرة لامريكا الشمالية ن...
5006,الغرض اتفاقية التجارة الحرة لامريكا الشمالية,الاقتصاد والعمل,الغرض اتفاقية التجارة الحرة لامريكا الشمالية ن...
5007,هدف اعجاز القران وفقا للمعتقد الاسلامي,الدين,يودي الاعجاز غرضين رييسين الاول اثبات اصالة ال...


In [82]:
has_numbers_q = df['QuestionText'].str.contains(r'\d', regex=True).sum()
has_numbers_q

np.int64(128)

In [83]:
has_english_q = df['QuestionText'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_q

np.int64(0)

In [84]:
has_english_a = df['Answer'].str.contains(r'[A-Za-z]', regex=True).sum()
has_english_a

np.int64(0)

In [85]:
remaining_q = df[df['QuestionText'].str.contains(r'[A-Za-z]', regex=True)]
remaining_a = df[df['Answer'].str.contains(r'[A-Za-z]', regex=True)]

remaining_q
remaining_a

,QuestionText,Category,Answer


In [86]:
remaining_q

,QuestionText,Category,Answer


# **Stemming**

In [87]:
stemmer=ISRIStemmer()
def stemming(text):
    words = text.split()  # split into words
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)  # return string

for col in Columns:
  df[col]=df[col].apply(stemming)
df

/tmp/ipykernel_390/4205959459.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col]=df[col].apply(stemming)


,QuestionText,Category,Answer
0,ايه فضل درس سبق وقت الحالي,التعليم,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,الس قطن عمد ثرة مصر,الاقتصاد والعمل,قطن عبر اهم نتج زرع مصر يعد عمد ريس قصد صري
2,صعد شمس شرق,التعليم,شمس صعد شرق
3,عرف كتر بان كين حية دقق,التعليم,كتر عرف بان كين حية دقق
4,ايت هوء سسا ترج,التعليم,هوء يتك سسا ترج
...,...,...,...
5004,حدث تسخ حدد,العلوم,تسخ حدد سرع حرك جزي يود مدد زيد حجم
5005,هدف تفق جار حرة لمر شمل,الاقتصاد والعمل,هدف تفق جار حرة لمر شمل نفت عزز جار ولي تحد كن...
5006,غرض تفق جار حرة لمر شمل,الاقتصاد والعمل,غرض تفق جار حرة لمر شمل نفت قلل عيق جرة عزز نف...
5007,هدف عجز قرن وفق عقد سلم,الدين,يود عجز غرض ريس اول اثب صلة قرن وصح كمصدر اله ...


# **Label Encoding**

In [88]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df['Category'])

/tmp/ipykernel_390/3925959778.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = label_encoder.fit_transform(df['Category'])


In [89]:
mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(mapping)

{'الاقتصاد والعمل': np.int64(0), 'البيئة والطاقة': np.int64(1), 'البيولوجيا': np.int64(2), 'التاريخ': np.int64(3), 'الترفيه': np.int64(4), 'التطوع': np.int64(5), 'التعليم': np.int64(6), 'التكنولوجيا': np.int64(7), 'الثقافة': np.int64(8), 'الجغرافيا': np.int64(9), 'الدين': np.int64(10), 'الرياضة': np.int64(11), 'السفر والسياحة': np.int64(12), 'السياسة والقانون': np.int64(13), 'الصحة': np.int64(14), 'العلوم': np.int64(15), 'علم الاجتماع': np.int64(16)}


In [90]:
df.head()

,QuestionText,Category,Answer,label
0,ايه فضل درس سبق وقت الحالي,التعليم,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث,6
1,الس قطن عمد ثرة مصر,الاقتصاد والعمل,قطن عبر اهم نتج زرع مصر يعد عمد ريس قصد صري,0
2,صعد شمس شرق,التعليم,شمس صعد شرق,6
3,عرف كتر بان كين حية دقق,التعليم,كتر عرف بان كين حية دقق,6
4,ايت هوء سسا ترج,التعليم,هوء يتك سسا ترج,6


# **Download Dataset**

In [91]:
df.to_csv("NLP_dataset.csv", index=False, encoding="utf-8-sig")

In [92]:
from google.colab import files
files.download("NLP_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>